<!--
---
PURPOSE: Convert pose predictions to features and motifs.
REQUIRES:
  - outputs/pose/session_{id}_pose_predictions.parquet
PRODUCES:
  - outputs/pose/session_{id}_pose_features.parquet
  - outputs/pose/session_{id}_motifs.parquet
WHAT_NEXT: notebooks/08_Neural_Behavior_Fusion_and_Modeling.ipynb
---
-->

# 07 Pose to Motifs Feature Engineering

**Purpose**
Convert pose predictions to features and motifs.

**Requires**
- `outputs/pose/session_{id}_pose_predictions.parquet`

**Produces**
- `outputs/pose/session_{id}_pose_features.parquet`
- `outputs/pose/session_{id}_motifs.parquet`

**What to run next**
- `notebooks/08_Neural_Behavior_Fusion_and_Modeling.ipynb`

We load pose predictions, derive features, and assign motifs.


## Environment Setup
We add the repo `src/` to the Python path so notebooks can import shared modules.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

## Prerequisite Check
We parse the notebook header and validate required artifacts before running downstream steps.

In [ ]:
from pathlib import Path
from reports import parse_notebook_header, validate_prerequisites

nb_path = Path.cwd() / "notebooks" / "07_Pose_to_Motifs_Feature_Engineering.ipynb"
header = parse_notebook_header(nb_path)
missing = validate_prerequisites(header.get("REQUIRES", []))
if missing:
    print("Missing prerequisites:")
    for item in missing:
        print(" -", item)
else:
    print("All prerequisites satisfied.")

## Step 1: Load pose predictions
If predictions are missing, create a small mock table to keep the pipeline runnable.

In [ ]:
from config import get_config, make_provenance
from io_sessions import load_sessions_csv
from timebase import write_parquet_with_timebase
from features_pose import derive_pose_features
from motifs import motifs_kmeans
from viz import plot_motif_transition
import pandas as pd
import numpy as np

cfg = get_config()
SESSION_ID = load_sessions_csv().session_id.tolist()[:1][0]
pose_path = cfg.outputs_dir / "pose" / f"session_{SESSION_ID}_pose_predictions.parquet"

if pose_path.exists():
    pose_df = pd.read_parquet(pose_path)
else:
    pose_df = pd.DataFrame({"frame_idx": np.arange(100), "t": np.linspace(0, 10, 100), "kp0_x": np.random.randn(100), "kp0_y": np.random.randn(100), "confidence": 1.0})

features = derive_pose_features(pose_df)
features_path = cfg.outputs_dir / "pose" / f"session_{SESSION_ID}_pose_features.parquet"
write_parquet_with_timebase(
    features,
    features_path,
    provenance=make_provenance(SESSION_ID, "nwb"),
    required_columns=["t"],
)

motifs_df = motifs_kmeans(features, n_clusters=6)
motifs_path = cfg.outputs_dir / "pose" / f"session_{SESSION_ID}_motifs.parquet"
write_parquet_with_timebase(
    motifs_df,
    motifs_path,
    provenance=make_provenance(SESSION_ID, "nwb"),
    required_columns=["t", "motif_id"],
)

plot_motif_transition(motifs_df)